# Assignment 4:  A Cognitive FAQ System Using Pandas (Nova 2.0)
Name : Akanksha Thakur
Roll No.: 1024160019

Q1: Build Your Personalized Knowledge Base

In [1]:
import pandas as pd

# Replace with your actual roll number
roll_number = "1024160019"  

# Extract last two digits
d1, d2 = int(roll_number[-2]), int(roll_number[-1])
categories = ["billing", "account", "general"]

# Map digits to categories using d % 3
cat1 = categories[d1 % 3]
cat2 = categories[d2 % 3]

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]

# Personalized entries tailored to computed categories
personalized_entries = [
    {"question": "how do i view my invoices", "answer": "Log into your account and check Billing > Invoices.", "keywords": "invoice bill statement download", "category": cat1},
    {"question": "how to change registered email", "answer": "Go to Account Settings > Profile > Email.", "keywords": "email profile update change", "category": cat2}
]

df = pd.DataFrame(fixed_entries + personalized_entries)
print(df)

                         question  \
0          what is the annual fee   
1           how to reset password   
2     what are your working hours   
3           how can i pay the fee   
4       how do i view my invoices   
5  how to change registered email   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Log into your account and check Billing > Invo...   
5          Go to Account Settings > Profile > Email.   

                          keywords category  
0            fee cost price charge  billing  
1             password reset login  account  
2           hours timing open time  general  
3              pay payment upi fee  billing  
4  invoice bill statement download  account  
5      email profile update change  billing  


Q2: Generate and Score a Hypothesis

In [2]:
def score_query(query: str, faq_df: pd.DataFrame) -> pd.DataFrame:
    query_words = set(query.lower().split())
    
    def calculate_score(row):
        # Match query against both keywords and question text
        words = set(row["keywords"].lower().split()).union(set(row["question"].lower().split()))
        return len(query_words.intersection(words))

    scored_df = faq_df.copy()
    scored_df["score"] = scored_df.apply(calculate_score, axis=1)
    
    # Filter non-zero matches and sort by score descending
    return scored_df[scored_df["score"] > 0].sort_values(by="score", ascending=False)

# Test query
print(score_query("pay fee", df)[["question", "score"]])

                 question  score
3   how can i pay the fee      2
0  what is the annual fee      1


Q3: Question Retrieval by Category

In [3]:
def same_category(category_name: str, faq_df: pd.DataFrame) -> pd.DataFrame:
    return faq_df[faq_df["category"] == category_name][["question", "category"]]

# Test using the category from the first personalized entry (cat1)
print(same_category(cat1, df))

                    question category
1      how to reset password  account
4  how do i view my invoices  account


Q4: Add Keyword and Save to CSV

In [4]:
# Roll number configuration
roll_number = "1024160019"

# Pick the first entry (index 0) and define the new keyword directly
entry_index = 0
new_keyword = "yearly"

# Append the new keyword to entry 0's existing keywords
df.loc[entry_index, "keywords"] += f" {new_keyword}"

# Save the updated DataFrame to CSV with your roll number
file_name = f"{roll_number}_faq_data.csv"
df.to_csv(file_name, index=False)

# Display result
print(f"Updated entry {entry_index} keywords: {df.loc[entry_index, 'keywords']}")
print(f"File saved as: {file_name}")

Updated entry 0 keywords: fee cost price charge yearly
File saved as: 1024160019_faq_data.csv


Q5: Count FAQ Entries per Category

In [5]:
category_counts = df.groupby("category").size()
print(category_counts)

category
account    2
billing    3
general    1
dtype: int64


Q6: Score Function with Tie Detection

In [6]:
def score_query_with_ties(query: str, faq_df: pd.DataFrame) -> pd.DataFrame:
    ranked = score_query(query, faq_df)
    
    if ranked.empty:
        print(f"No match found for query: '{query}'")
        return ranked
    
    max_score = ranked["score"].max()
    top_matches = ranked[ranked["score"] == max_score]
    
    if len(top_matches) > 1:
        print(f"Tie detected! {len(top_matches)} entries matched with score {max_score}:")
    else:
        print(f"Single best match with score {max_score}:")
        
    return top_matches[["question", "category", "score"]]

# Demonstration 1: Query producing a tie (matches both fee entries)
print("--- Query with Tie ---")
print(score_query_with_ties("fee", df))

# Demonstration 2: Query producing a single top match
print("\n--- Query without Tie ---")
print(score_query_with_ties("reset password", df))

--- Query with Tie ---
Tie detected! 2 entries matched with score 1:
                 question category  score
0  what is the annual fee  billing      1
3   how can i pay the fee  billing      1

--- Query without Tie ---
Single best match with score 2:
                question category  score
1  how to reset password  account      2
